# Start Here: Qiskit Basics -- Entanglement on 9 Qubits

This notebook is a hands-on primer before diving into the quantum error
correction examples in `examples/`. It builds a 9-qubit register (the same
size as the surface code's 9 data qubits used later in this project) and
walks through four qualitatively different ways to entangle it:

1. **No entanglement** -- a product state, as a baseline.
2. **Bell pairs** -- several independent, maximally-entangled *pairs*.
3. **GHZ state** -- one big "cat state" entangling all 9 qubits together.
4. **Cluster / graph state** -- nearest-neighbor entanglement on a 3x3 grid,
   the same lattice topology the surface code uses.

Each code cell is preceded by a short explanation of what it does and, for
imports, what the imported names actually give you. Circuits are printed
with Qiskit's own drawer (`QuantumCircuit.draw`) so you can see exactly
what gates are being applied.

## Imports

### Circuit building blocks

- `QuantumCircuit` is the object you build a circuit on: you call methods
  like `.h(qubit)` or `.cx(control, target)` on it to append gates.
- `QuantumRegister` is a named group of qubits. Using one (instead of a
  bare `QuantumCircuit(9)`) just makes circuit diagrams more readable --
  you'll see labels like `q_0` instead of an unnamed wire.

We won't need `ClassicalRegister` here (that's for reading out measurement
results into classical bits, used heavily in `examples/` for syndrome
readout) -- this notebook inspects the quantum state directly instead of
measuring it, so there's nothing to import for that yet.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister

### Inspecting what a circuit actually did

Building a circuit only describes *what to do*; to see the resulting
quantum state we need some analysis tools from `qiskit.quantum_info`:

- `Statevector` -- wraps a state vector and knows how to evolve through a
  circuit (`Statevector.from_instruction(qc)`), doing exact linear-algebra
  simulation. No shots, no noise -- just the true amplitudes. (The later
  examples in `examples/` use `qiskit_aer.AerSimulator` instead, because
  they need mid-circuit measurement and classical feed-forward, which
  `Statevector` alone can't do -- but for the plain circuits in this
  notebook, `Statevector` is simpler and sufficient.)
- `partial_trace` -- given a state over several qubits, discards ("traces
  out") a chosen subset and returns the resulting density matrix on the
  rest. This is the standard tool for asking "what does qubit i look like
  on its own, ignoring everyone else?" -- and the answer can be a *mixed*
  state even though the full 9-qubit state is pure, which is precisely the
  signature of entanglement.
- `entropy` -- the von Neumann entropy of a density matrix. For a single
  qubit traced out of a larger pure state, this is 0 if that qubit is
  unentangled with the rest, and 1 (bit) if it's maximally entangled.
- `concurrence` -- a standard entanglement measure for a *two-qubit*
  state (0 = not entangled, 1 = maximally entangled). We'll use it on
  2-qubit reduced states to map out *which pairs* of qubits are actually
  entangled with each other, not just *whether* a qubit is entangled with
  something.

In [ ]:
from qiskit.quantum_info import Statevector, partial_trace, entropy, concurrence

### Numerics and plotting

`numpy` for array/matrix math, `matplotlib` to draw circuits (`qc.draw('mpl')`)
and our own entanglement heatmaps and bar charts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

N = 9  # qubits -- matching the surface code's 9 data qubits used later in this project

## A couple of small helpers

Rather than repeat the same "trace out everyone except qubit i" or
"trace out everyone except qubits i, j" code for every example below, two
small helper functions. `partial_trace`'s `qargs` argument is the list of
qubits to **discard**, so `reduced_state` just computes the complement of
whatever you want to keep.

In [ ]:
def reduced_state(full_state, keep):
    """Density matrix on qubits in `keep`, tracing out everything else."""
    keep = set(keep)
    trace_out = [q for q in range(full_state.num_qubits) if q not in keep]
    return partial_trace(full_state, trace_out)


def single_qubit_entropies(full_state, n=N):
    """Entanglement entropy (bits) of each qubit with the rest of the register."""
    return [entropy(reduced_state(full_state, [i]), base=2) for i in range(n)]


def pairwise_concurrence_matrix(full_state, n=N):
    """n x n matrix of pairwise concurrence between every pair of qubits."""
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            c = concurrence(reduced_state(full_state, [i, j]))
            mat[i, j] = mat[j, i] = c
    return mat

## 1. No entanglement -- a product state baseline

Each qubit gets its own independent rotation (`Ry` by a different random
angle) and nothing else touches it. There are no two-qubit gates in this
circuit at all, so by construction no qubit can be correlated with any
other -- this is our "entanglement = 0" reference point for everything
that follows.

In [ ]:
rng = np.random.default_rng(0)

qc_none = QuantumCircuit(QuantumRegister(N, "q"), name="no_entanglement")
for i in range(N):
    qc_none.ry(rng.uniform(0, np.pi), i)

print(qc_none.draw("text"))
qc_none.draw("mpl")

In [ ]:
sv_none = Statevector.from_instruction(qc_none)
print("single-qubit entanglement entropy (bits) per qubit:")
print(np.round(single_qubit_entropies(sv_none), 3))

As expected: all zeros. Numerically confirms what's obvious from the
circuit -- nobody is entangled with anybody.

## 2. Bell pairs -- several independent, local entangled pairs

The standard Bell-pair recipe is `H` on one qubit then `CX` from it to a
partner. We do this four times, on qubits (0,1), (2,3), (4,5), (6,7),
leaving qubit 8 alone (untouched, so it stays in `|0>`). Each pair becomes
maximally entangled *with its own partner only* -- there is no gate
connecting different pairs, so (0,1) and (2,3) are completely independent
of each other.

In [ ]:
qc_bell = QuantumCircuit(QuantumRegister(N, "q"), name="bell_pairs")
pairs = [(0, 1), (2, 3), (4, 5), (6, 7)]
for a, b in pairs:
    qc_bell.h(a)
    qc_bell.cx(a, b)
# qubit 8 is left alone -- not entangled with anything

print(qc_bell.draw("text"))
qc_bell.draw("mpl")

In [ ]:
sv_bell = Statevector.from_instruction(qc_bell)
print("single-qubit entanglement entropy (bits) per qubit:")
print(np.round(single_qubit_entropies(sv_bell), 3))

Every qubit in a pair shows 1 bit of entanglement entropy (maximally mixed
on its own) -- except qubit 8, which shows 0, exactly as expected since we
never touched it.

## 3. GHZ state -- one entangled state across all 9 qubits

`H` on qubit 0, then a chain of `CX` gates (0->1->2->...->8) spreads that
superposition across the whole register:
$(|000000000\rangle + |111111111\rangle) / \sqrt{2}$.
Every qubit is now entangled with every other qubit -- there's no subgroup
you can point to that's independent of the rest, unlike the Bell-pair case.

In [ ]:
qc_ghz = QuantumCircuit(QuantumRegister(N, "q"), name="ghz")
qc_ghz.h(0)
for i in range(N - 1):
    qc_ghz.cx(i, i + 1)

print(qc_ghz.draw("text"))
qc_ghz.draw("mpl")

In [ ]:
sv_ghz = Statevector.from_instruction(qc_ghz)
print("single-qubit entanglement entropy (bits) per qubit:")
print(np.round(single_qubit_entropies(sv_ghz), 3))

Every qubit shows 1 bit of entropy too -- the *same* number as a Bell
pair! Single-qubit entropy alone can't tell GHZ and Bell pairs apart. We
need to look at *pairwise* structure -- see the comparison section below.

## 4. Cluster (graph) state on a 3x3 grid

Instead of one big chain, put every qubit into `|+>` (`H`), then apply
`CZ` between qubits that are *nearest neighbors on a 3x3 grid* -- the same
lattice used by the surface code's 9 data qubits in this project
(`qec/surface_code.py`'s `DATA_POS`). This is a "graph state": the
entanglement pattern directly mirrors the grid's edges, unlike GHZ (one
global chain) or Bell pairs (isolated dominoes).

In [ ]:
# grid layout (same indexing as qec/surface_code.py's DATA_POS):
#   0 1 2
#   3 4 5
#   6 7 8
GRID_EDGES = [
    (0, 1), (1, 2),
    (3, 4), (4, 5),
    (6, 7), (7, 8),
    (0, 3), (3, 6),
    (1, 4), (4, 7),
    (2, 5), (5, 8),
]

qc_cluster = QuantumCircuit(QuantumRegister(N, "q"), name="cluster_grid")
for i in range(N):
    qc_cluster.h(i)
for a, b in GRID_EDGES:
    qc_cluster.cz(a, b)

print(qc_cluster.draw("text"))
qc_cluster.draw("mpl")

In [ ]:
sv_cluster = Statevector.from_instruction(qc_cluster)
print("single-qubit entanglement entropy (bits) per qubit:")
print(np.round(single_qubit_entropies(sv_cluster), 3))

Still 1 bit everywhere (every qubit has at least one grid neighbor) -- so
again indistinguishable from GHZ / Bell pairs by this metric alone. The
real difference is in *which pairs* are entangled, not *how much* any
single qubit is -- exactly what the next section shows.

## Telling them apart: pairwise concurrence

Single-qubit entropy shows *that* a qubit is entangled with the rest of
the register, but not *with whom*. `concurrence` on a 2-qubit reduced
state tells us exactly that. Computing it for every pair (i, j) and
plotting as a heatmap should reveal each state's actual structure:

- **No entanglement**: all zero.
- **Bell pairs**: exactly 4 bright off-diagonal cells, at (0,1), (2,3),
  (4,5), (6,7) -- nothing else lit up.
- **Cluster / grid state**: cells lit up along the grid's edges.
- **GHZ**: let's find out.

In [ ]:
states = {
    "no entanglement": sv_none,
    "Bell pairs": sv_bell,
    "GHZ": sv_ghz,
    "cluster (3x3 grid)": sv_cluster,
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
im = None
for ax, (name, sv) in zip(axes, states.items()):
    mat = pairwise_concurrence_matrix(sv)
    im = ax.imshow(mat, vmin=0, vmax=1, cmap="viridis")
    ax.set_title(name)
    ax.set_xlabel("qubit")
    ax.set_ylabel("qubit")
fig.colorbar(im, ax=axes, label="pairwise concurrence", shrink=0.7)
plt.show()

for name, sv in states.items():
    mat = pairwise_concurrence_matrix(sv)
    print(f"{name:22s} max pairwise concurrence = {mat.max():.3f}")

## The GHZ surprise

Notice the GHZ heatmap is basically **all zero** -- no pair of qubits
shows any pairwise concurrence at all, even though every single qubit
individually has 1 bit of entanglement entropy (section 3). This isn't a
bug: it's a real, well-known feature of GHZ states. Their entanglement is
genuinely *multipartite* -- it can't be squeezed into any 2-qubit
sub-correlation, a consequence of entanglement monogamy. If you want
entanglement you can "see" between specific pairs, you need something
like the Bell-pair or cluster-state construction, not a GHZ chain.

This is exactly why the surface code (built with a grid-graph-like
stabilizer structure -- see `qec/surface_code.py`) and the repetition
codes (built with local pairwise-style CNOTs -- see `qec/circuits.py`)
look structurally more like the Bell-pair / cluster-state examples above
than like a GHZ state, even though a naive "N qubits become entangled"
picture might suggest otherwise.

## Next steps

Now that you've seen how to build, draw, and quantitatively inspect
entangled circuits in Qiskit, head to `examples/01_bit_flip_basic.py`
onward: those scripts encode a *single* logical qubit into several
physical qubits (an entangled state much like the cluster/Bell examples
here), inject random errors, and use syndrome measurement + classical
feed-forward correction to protect it. See the project `README.md` for
the full rundown of what each example demonstrates.